In [47]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import glob

# Caminhos para os arquivos de treino e itens
train_files = sorted(glob.glob("challenge-webmedia-e-globo-2023/files/treino/treino_parte*.csv"))
item_files = sorted(glob.glob("challenge-webmedia-e-globo-2023/itens/itens/itens-parte*.csv"))

# Ler e concatenar os arquivos de treino
df_train = pd.concat([pd.read_csv(file) for file in train_files], ignore_index=True)

# Ler e concatenar os arquivos de itens
df_items = pd.concat([pd.read_csv(file) for file in item_files], ignore_index=True)

In [2]:
df_train.head()

,userId,userType,historySize,history,timestampHistory,numberOfClicksHistory,timeOnPageHistory,scrollPercentageHistory,pageVisitsCountHistory,timestampHistory_new
0,f98d1132f60d46883ce49583257104d15ce723b3bbda21...,Non-Logged,3,"c8aab885-433d-4e46-8066-479f40ba7fb2, 68d2039c...","1657146417045, 1657146605778, 1657146698738","76, 38, 41","20380, 21184, 35438","50.3, 18.18, 16.46","2, 1, 1","1657146417045, 1657146605778, 1657146698738"
1,2c1080975e257ed630e26679edbe4d5c850c65f3e09f65...,Non-Logged,60,"3325b5a1-979a-4cb3-82b6-63905c9edbe8, fe856057...","1656684240278, 1656761266729, 1656761528085, 1...","7, 80, 2, 1, 7, 62, 26, 44, 4, 4, 14, 45, 13, ...","6049, 210489, 8672, 10000, 30000, 123007, 9965...","25.35, 45.66, 35.3, 28.05, 36.53, 47.57, 55.33...","1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1...","1656684240278, 1656761266729, 1656761528085, 1..."
2,0adffd7450d3b9840d8c6215f0569ad942e782fb19b805...,Logged,107,"04756569-593e-4133-a95a-83d35d43dbbd, 29b6b142...","1656678946256, 1656701076495, 1656701882565, 1...","0, 0, 0, 0, 0, 44, 0, 0, 2, 1, 0, 0, 0, 44, 0,...","311274, 140000, 32515, 157018, 118689, 159243,...","67.58, 47.22, 41.52, 63.09, 51.38, 65.11, 71.9...","1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1...","1656678946256, 1656701076495, 1656701882565, 1..."
3,c1e8d644329a78ea1f994292db624c57980b2886cfbc2d...,Non-Logged,56,"1f2b9c2f-a2d2-4192-b009-09065da8ec23, 04756569...","1658333312180, 1658404553818, 1658408449062, 1...","8, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 2, 0, 1, 1...","182696, 91925, 30000, 273655, 126409, 42980, 1...","58.26, 72.66, 22.57, 59.89, 40.36, 36.35, 14.7...","1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1...","1658333312180, 1658404553818, 1658408449062, 1..."
4,e777d1f31d4d955b63d60acc13df336d3903f52ab8f8f4...,Non-Logged,4,"bebdeb3e-1699-43e0-a1b8-989f5a6ab679, f4b484a7...","1658766608801, 1658766608801, 1660084035094, 1...","579, 579, 7, 2","801396, 801396, 10000, 10000","78.74, 78.74, 16.71, 9.34","7, 7, 1, 1","1658766608801, 1658766608801, 1660084035094, 1..."


In [3]:
df_items.head()

,page,url,issued,modified,title,body,caption
0,13db0ab1-eea2-4603-84c4-f40a876c7400,http://g1.globo.com/am/amazonas/noticia/2022/0...,2022-06-18 20:37:45+00:00,2023-04-15 00:02:08+00:00,Caso Bruno e Dom: 3º suspeito tem prisão tempo...,"Após audiência de custódia, a Justiça do Amazo...",Jeferson da Silva Lima foi escoltado por agent...
1,92907b73-5cd3-4184-8d8c-e206aed2bf1c,http://g1.globo.com/pa/santarem-regiao/noticia...,2019-06-20 17:19:52+00:00,2023-06-16 20:19:15+00:00,Linguajar dos santarenos é diferenciado e chei...,Vista aérea de Santarém\nÁdrio Denner/ AD Prod...,As expressões santarenas não significam apenas...
2,61e07f64-cddf-46f2-b50c-ea0a39c22050,http://g1.globo.com/mundo/noticia/2022/07/08/e...,2022-07-08 08:55:52+00:00,2023-04-15 04:25:39+00:00,Ex-premiê Shinzo Abe morre após ser baleado no...,Novo vídeo mostra que assassino de Shinzo Abe ...,Ex-primeiro-ministro foi atingido por tiros de...
3,30e2e6c5-554a-48ed-a35f-6c6691c8ac9b,http://g1.globo.com/politica/noticia/2021/09/0...,2021-09-09 19:06:46+00:00,2023-06-07 17:44:54+00:00,"Relator no STF, Fachin vota contra marco tempo...","Relator no STF, Fachin vota contra marco tempo...",Ministro defendeu que posse indígena é diferen...
4,9dff71eb-b681-40c7-ac8d-68017ac36675,http://g1.globo.com/politica/noticia/2021/09/1...,2021-09-15 19:16:13+00:00,2023-06-07 17:43:39+00:00,"\nApós 2 votos, pedido de vista suspende julga...",Após um pedido de vista (mais tempo para análi...,"Pelo marco temporal, índios só podem reivindic..."


In [4]:
df_train["history"][0].split(", ")

['c8aab885-433d-4e46-8066-479f40ba7fb2',
 '68d2039c-c9aa-456c-ac33-9b2e8677fba7',
 '13e423ce-1d69-4c78-bc18-e8c8f7271964']

In [ ]:
teste = df_train["history"].str.split(", ")
teste.explode().value_counts()

In [ ]:
# Criar um ranking de popularidade baseado no número de cliques
popular_articles = df_train["history"].str.split(", ").explode().value_counts().index.tolist()
popular_articles

In [ ]:
# Criar embeddings das notícias usando TF-IDF
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop = stopwords.words("portuguese")
stop


In [49]:
vectorizer = TfidfVectorizer(stop_words=stop)
tfidf_matrix = vectorizer.fit_transform(df_items['title'] + " " + df_items['body'])

In [ ]:
# Criar um dicionário para mapear páginas aos índices
doc_indices = {page: idx for idx, page in enumerate(df_items['page'])}
doc_indices

In [51]:
doc_indices["c8aab885-433d-4e46-8066-479f40ba7fb2"]

193275

In [61]:
def recommend_news(user_history, top_n=5):
    if not user_history:  # Cold start (usuário novo)
        return df_items[df_items['page'].isin(popular_articles[:top_n])][['page', 'title']]
    
    # Obter índices das notícias que o usuário viu
    viewed_indices = [doc_indices[page] for page in user_history if page in doc_indices]
    print(viewed_indices)
    if not viewed_indices:
        return df_items[df_items['page'].isin(popular_articles[:top_n])][['page', 'title']]
    
    # Calcular similaridade entre as notícias vistas e todas as outras
    user_profile = np.mean(tfidf_matrix[viewed_indices], axis=0)
    scores = cosine_similarity(np.asarray(user_profile), tfidf_matrix).flatten()
    
    # Obter as top-N notícias mais similares que o usuário ainda não viu
    recommended_indices = np.argsort(scores)[::-1]
    recommended_pages = [df_items.iloc[i]['page'] for i in recommended_indices if df_items.iloc[i]['page'] not in user_history]

    return (df_items[df_items['page'].isin(recommended_pages[:top_n])][['page', 'title']] ,df_items[df_items["page"].isin(user_history)][['page', 'title']] )

# Testar com um usuário aleatório
sample_user_history = df_train.iloc[0]['history'].split(", ")
sample_user_history

['c8aab885-433d-4e46-8066-479f40ba7fb2',
 '68d2039c-c9aa-456c-ac33-9b2e8677fba7',
 '13e423ce-1d69-4c78-bc18-e8c8f7271964']

In [ ]:
recommendations = recommend_news(sample_user_history, top_n=5)
recommendations

In [63]:
recommendations[0]

,page,title
36243,d209e890-58be-47a8-8813-ff1b275eb400,Caminhoneira 'Musa das Estradas' posta última ...
89487,54c126a4-25fd-4b71-80cb-8ab355956168,Caminhoneira 'Musa das Estradas' mostra rosto ...
121826,a12a1004-93d7-4131-a8b9-89e4512d803b,Caminhoneira 'Musa das Estradas' faz vídeo em ...
172371,5391514a-8bb1-4388-be9b-63d7c278f151,"Em vídeo, caminhoneira 'Musa das Estradas' mos..."
228763,32ec3a09-bd8c-4744-93e6-69fa91d4e5b0,Caminhoneira 'Musa das Estradas' recebe alta a...


In [64]:
recommendations[1]

,page,title
158881,68d2039c-c9aa-456c-ac33-9b2e8677fba7,'Mulher-Gato' foi proibida de entrar na Maré a...
193275,c8aab885-433d-4e46-8066-479f40ba7fb2,"Você viu? 'Musa das Estradas' faz vídeo de pé,..."
203270,13e423ce-1d69-4c78-bc18-e8c8f7271964,Caminhoneira 'Musa das Estradas' mostra rosto ...


In [13]:
import pickle
import joblib

# Salvar TF-IDF Vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Salvar Matriz TF-IDF
joblib.dump(tfidf_matrix, "tfidf_matrix.pkl")

# Salvar Dicionário de Índices
with open("doc_indices.pkl", "wb") as f:
    pickle.dump(doc_indices, f)
